In [ ]:
%pip install google-api-python-client pandas openpyxl python-dotenv

In [15]:
# Import, konfigurasi, dan fungsi autentikasi
import io
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth.exceptions import RefreshError
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

load_dotenv("../.env")
FOLDER_ID = os.getenv("GDRIVE_FOLDER_ID", "1kRAGRIc2uLg1v4RCUWr4Hh345oiYXefS")
TARGET_SHEET = "REKAPITULASI"

CREDENTIALS_FILE = Path("../credentials.json")  # client secret dari Google Cloud (jangan dihapus)
TOKEN_FILE = Path("../token.json")              # dibuat otomatis, boleh dihapus kapan saja
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

SHEET_MIME = "application/vnd.google-apps.spreadsheet"
FOLDER_MIME = "application/vnd.google-apps.folder"
XLSX_MIME = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"

assert CREDENTIALS_FILE.exists(), f"credentials.json tidak ditemukan di {CREDENTIALS_FILE.resolve()}"


def get_credentials():
    """Ambil kredensial OAuth. Token di-refresh otomatis; login ulang kalau refresh gagal."""
    creds = None
    if TOKEN_FILE.exists():
        creds = Credentials.from_authorized_user_file(str(TOKEN_FILE), SCOPES)

    # 1. Token masih valid
    if creds and creds.valid:
        return creds

    # 2. Token kedaluwarsa -> refresh dan simpan token baru
    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
            TOKEN_FILE.write_text(creds.to_json())
            print("Token berhasil diperbarui")
            return creds
        except RefreshError as e:
            # 3. Refresh gagal (token dicabut / kedaluwarsa permanen) -> hapus dan login ulang
            print(f"Refresh token gagal ({e}). Menghapus token.json dan login ulang...")
            TOKEN_FILE.unlink(missing_ok=True)

    # Login baru lewat browser
    flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS_FILE), SCOPES)
    creds = flow.run_local_server(port=0)
    TOKEN_FILE.write_text(creds.to_json())
    print("Login berhasil, token.json dibuat")
    return creds

In [16]:
# Koneksi ke Google Drive API
creds = get_credentials()
service = build("drive", "v3", credentials=creds, cache_discovery=False)

# Tes koneksi: tampilkan akun yang sedang login
about = service.about().get(fields="user(displayName, emailAddress)").execute()
print("Terhubung sebagai:", about["user"]["emailAddress"])

Terhubung sebagai: erisalfionita18@gmail.com


In [17]:
# Daftar file di folder 
def list_files(folder_id):
    results, page_token = [], None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id, name, mimeType, modifiedTime)",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        for f in resp.get("files", []):
            if f["mimeType"] == FOLDER_MIME:
                results.extend(list_files(f["id"]))
            elif f["mimeType"] in (SHEET_MIME, XLSX_MIME):
                results.append(f)
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return results


files = sorted(list_files(FOLDER_ID), key=lambda f: f["name"])
df_files = pd.DataFrame(files)
print(f"Jumlah file ditemukan: {len(files)}")
df_files

Jumlah file ditemukan: 44


,mimeType,id,name,modifiedTime
0,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,106JMAml_2dmAJ3n2z0I9MbKQvGmUBgLT,01. REKAP JANUARI 2023.xlsx,2026-10-02T07:59:12.334Z
1,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1muD9pjEAru_OQBoaFnBnbEKP8yscpIcH,01. REKAP JANUARI 2024.xlsx,2025-01-02T03:12:02.000Z
2,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1IIuOzXjKdyPRZnv0FZXIzaksoLQJBLYN,01. REKAPITULASI JANUARI 2025.xlsx,2026-10-06T07:16:44.563Z
3,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1Cc7xxjDSKj04td4I-2_iZWtMOAhF8WOU,01. REKAPITULASI JANUARI 2026.xlsx,2026-10-05T08:39:19.606Z
4,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1pMJddHvgyEjIa4BsBsi1oHZWMbEiT1St,02. REKAP FEBRUARI 2023.xlsx,2026-11-04T19:30:58.785Z
5,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,10JZfNDXa1g6-6FVPfHtsOwjyzXIvsOqF,02. REKAP FEBRUARI 2024.xlsx,2024-02-29T08:02:31.000Z
6,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1oiRmVHlYJPtrt4b9ckPwPIabpkCQu1aB,02. REKAPITULASI FEBRUARI 2025.xlsx,2026-10-05T03:39:07.338Z
7,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1Jvl7qZAZiASkrltOtkFjTSSjNEAcIaWe,02. REKAPITULASI FEBRUARI 2026.xlsx,2026-10-06T02:58:32.675Z
8,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1u267vQBEWRGXS6TLlLwEbEsWPro21oXU,03. REKAP MARET 2023.xlsx,2026-09-24T06:28:21.925Z
9,application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,1Mv3FjdrfcSdFqDTMPDdeL6cWDxqwvPLS,03. REKAP MARET 2024.xlsx,2026-10-08T02:13:20.884Z


In [18]:
# Folder bantuan untuk menyimpan file 
import re

XLSX_DIR = RAW_DIR / "xlsx_asli"        # salinan file asli dari Drive
REKAP_DIR = RAW_DIR / "rekapitulasi"    # CSV tab rekapitulasi
XLSX_DIR.mkdir(parents=True, exist_ok=True)
REKAP_DIR.mkdir(parents=True, exist_ok=True)


def safe_stem(name):
    """Nama file aman untuk Windows: buang ekstensi dan karakter terlarang."""
    stem = Path(name).stem if name.lower().endswith((".xlsx", ".xls", ".xlsm")) else name
    stem = re.sub(r'[<>:"/\\|?*]', "_", stem).strip().rstrip(".")
    return stem or "tanpa_nama"


def build_local_names(file_list):
    """Petakan file_id -> nama lokal unik. Kalau nama kembar, tambahkan 6 karakter awal ID."""
    counts = {}
    for f in file_list:
        counts[safe_stem(f["name"])] = counts.get(safe_stem(f["name"]), 0) + 1

    names = {}
    for f in file_list:
        stem = safe_stem(f["name"])
        names[f["id"]] = f"{stem}_{f['id'][:6]}" if counts[stem] > 1 else stem
    return names


def download_xlsx(file):
    """Download ke memori. Google Sheets di-export ke xlsx, file xlsx diunduh langsung."""
    if file["mimeType"] == SHEET_MIME:
        request = service.files().export_media(fileId=file["id"], mimeType=XLSX_MIME)
    else:
        request = service.files().get_media(fileId=file["id"], supportsAllDrives=True)

    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    return buffer


def find_rekap_sheets(sheet_names):
    """Cari tab rekapitulasi. Utama: mengandung 'rekapitulasi'. Cadangan: mengandung 'rekap'."""
    primary = [s for s in sheet_names if "rekapitulasi" in s.strip().lower()]
    if primary:
        return primary
    return [s for s in sheet_names if "rekap" in s.strip().lower()]


local_names = build_local_names(files)
print(f"Folder xlsx asli : {XLSX_DIR.resolve()}")
print(f"Folder rekap CSV : {REKAP_DIR.resolve()}")
print(f"Jumlah file      : {len(local_names)}")
print("Contoh nama lokal:", list(local_names.values())[:3])

Folder xlsx asli : D:\STUDY\SEMESTER 5\magang\proyek\dashboard-analitik-harga-komoditas-surabaya\data\raw\xlsx_asli
Folder rekap CSV : D:\STUDY\SEMESTER 5\magang\proyek\dashboard-analitik-harga-komoditas-surabaya\data\raw\rekapitulasi
Jumlah file      : 44
Contoh nama lokal: ['01. REKAP JANUARI 2023', '01. REKAP JANUARI 2024', '01. REKAPITULASI JANUARI 2025']


In [19]:
# Unduh file
from time import sleep

download_log = []

for i, f in enumerate(files, start=1):
    local_name = local_names[f["id"]]
    path = XLSX_DIR / f"{local_name}.xlsx"

    # Lewati kalau sudah pernah diunduh (aman dijalankan ulang)
    if path.exists() and path.stat().st_size > 0:
        download_log.append({"file": f["name"], "path": str(path), "status": "SUDAH ADA", "ukuran_kb": round(path.stat().st_size / 1024, 1)})
        continue

    try:
        buf = download_xlsx(f)
        path.write_bytes(buf.getvalue())
        download_log.append({"file": f["name"], "path": str(path), "status": "OK", "ukuran_kb": round(path.stat().st_size / 1024, 1)})
        print(f"[{i}/{len(files)}] OK  : {f['name']}")
    except Exception as e:
        download_log.append({"file": f["name"], "path": None, "status": f"ERROR: {e}", "ukuran_kb": None})
        print(f"[{i}/{len(files)}] GAGAL: {f['name']} -> {e}")
    sleep(0.2)  # jeda kecil agar tidak kena rate limit

df_download = pd.DataFrame(download_log)
df_download.to_csv(RAW_DIR / "_download_log.csv", index=False)

print("\nRingkasan:")
print(df_download["status"].str.split(":").str[0].value_counts().to_string())
df_download


Ringkasan:
status
SUDAH ADA    44


,file,path,status,ukuran_kb
0,01. REKAP JANUARI 2023.xlsx,..\data\raw\xlsx_asli\01. REKAP JANUARI 2023.xlsx,SUDAH ADA,650.2
1,01. REKAP JANUARI 2024.xlsx,..\data\raw\xlsx_asli\01. REKAP JANUARI 2024.xlsx,SUDAH ADA,3498.0
2,01. REKAPITULASI JANUARI 2025.xlsx,..\data\raw\xlsx_asli\01. REKAPITULASI JANUARI 2025.xlsx,SUDAH ADA,1245.5
3,01. REKAPITULASI JANUARI 2026.xlsx,..\data\raw\xlsx_asli\01. REKAPITULASI JANUARI 2026.xlsx,SUDAH ADA,457.8
4,02. REKAP FEBRUARI 2023.xlsx,..\data\raw\xlsx_asli\02. REKAP FEBRUARI 2023.xlsx,SUDAH ADA,796.0
5,02. REKAP FEBRUARI 2024.xlsx,..\data\raw\xlsx_asli\02. REKAP FEBRUARI 2024.xlsx,SUDAH ADA,708.7
6,02. REKAPITULASI FEBRUARI 2025.xlsx,..\data\raw\xlsx_asli\02. REKAPITULASI FEBRUARI 2025.xlsx,SUDAH ADA,888.5
7,02. REKAPITULASI FEBRUARI 2026.xlsx,..\data\raw\xlsx_asli\02. REKAPITULASI FEBRUARI 2026.xlsx,SUDAH ADA,318.5
8,03. REKAP MARET 2023.xlsx,..\data\raw\xlsx_asli\03. REKAP MARET 2023.xlsx,SUDAH ADA,783.6
9,03. REKAP MARET 2024.xlsx,..\data\raw\xlsx_asli\03. REKAP MARET 2024.xlsx,SUDAH ADA,1370.0


In [20]:
# Pengecualian manual: file yang punya lebih dari satu tab "rekapitulasi".
# Format: "nama file di Drive": "nama tab yang dipakai"
TAB_OVERRIDE = {
    "07. REKAP JULI 2023.xlsx": "REKAPITULASI BULANAN",
}

rows = []

for f in files:
    local_name = local_names[f["id"]]
    path = XLSX_DIR / f"{local_name}.xlsx"

    row = {
        "file": f["name"],
        "local_name": local_name,
        "tipe": "Google Sheets" if f["mimeType"] == SHEET_MIME else "Excel",
        "tab_rekapitulasi": None,
        "status": None,
        "semua_tab": None,
    }
    try:
        sheet_names = pd.ExcelFile(path).sheet_names
        row["semua_tab"] = " | ".join(sheet_names)

        if f["name"] in TAB_OVERRIDE:
            tab = TAB_OVERRIDE[f["name"]]
            if tab in sheet_names:
                row["tab_rekapitulasi"], row["status"] = tab, "OK"
            else:
                row["status"] = f"ERROR: tab override '{tab}' tidak ada di file"
        else:
            matches = find_rekap_sheets(sheet_names)
            if len(matches) == 1:
                row["tab_rekapitulasi"], row["status"] = matches[0], "OK"
            elif len(matches) > 1:
                row["tab_rekapitulasi"], row["status"] = " | ".join(matches), "LEBIH DARI 1"
            else:
                row["status"] = "TIDAK ADA"
    except Exception as e:
        row["status"] = f"ERROR: {e}"
    rows.append(row)

df_inventory = pd.DataFrame(rows)
df_inventory.to_csv(RAW_DIR / "_inventaris_file.csv", index=False)

print(f"Total file: {len(df_inventory)}")
print(df_inventory["status"].str.split(":").str[0].value_counts().to_string())

pd.set_option("display.max_colwidth", None)
df_inventory[df_inventory["status"] != "OK"]

Total file: 44
status
OK    44


,file,local_name,tipe,tab_rekapitulasi,status,semua_tab


In [21]:
# ubah xlxs ke csv 
extract_log = []

for _, r in df_inventory.iterrows():
    log = {
        "file": r["file"],
        "local_name": r["local_name"],
        "tab": r["tab_rekapitulasi"],
        "status": None,
        "baris": None,
        "kolom": None,
        "csv_path": None,
    }

    # Hanya proses file yang tab rekapitulasinya jelas (tepat satu)
    if r["status"] != "OK":
        log["status"] = f"DILEWATI ({r['status']})"
        extract_log.append(log)
        continue

    try:
        xlsx_path = XLSX_DIR / f"{r['local_name']}.xlsx"
        df = pd.read_excel(xlsx_path, sheet_name=r["tab_rekapitulasi"], header=None)

        log["baris"], log["kolom"] = df.shape

        # Tab yang isinya kosong semua ditandai, tidak disimpan
        if df.dropna(how="all").empty:
            log["status"] = "KOSONG"
        else:
            csv_path = REKAP_DIR / f"{r['local_name']}.csv"
            df.to_csv(csv_path, index=False, header=False, encoding="utf-8-sig")
            log["csv_path"] = str(csv_path)
            log["status"] = "OK"
    except Exception as e:
        log["status"] = f"ERROR: {e}"

    extract_log.append(log)

df_extract = pd.DataFrame(extract_log)
df_extract.to_csv(RAW_DIR / "_extract_log.csv", index=False)

print(f"Total file diproses: {len(df_extract)}")
print(df_extract["status"].str.split(":").str[0].value_counts().to_string())
df_extract

Total file diproses: 44
status
OK    44


,file,local_name,tab,status,baris,kolom,csv_path
0,01. REKAP JANUARI 2023.xlsx,01. REKAP JANUARI 2023,REKAPITULASI,OK,82,37,..\data\raw\rekapitulasi\01. REKAP JANUARI 2023.csv
1,01. REKAP JANUARI 2024.xlsx,01. REKAP JANUARI 2024,REKAPITULASI,OK,95,39,..\data\raw\rekapitulasi\01. REKAP JANUARI 2024.csv
2,01. REKAPITULASI JANUARI 2025.xlsx,01. REKAPITULASI JANUARI 2025,REKAPITULASI,OK,96,37,..\data\raw\rekapitulasi\01. REKAPITULASI JANUARI 2025.csv
3,01. REKAPITULASI JANUARI 2026.xlsx,01. REKAPITULASI JANUARI 2026,REKAPITULASI,OK,96,35,..\data\raw\rekapitulasi\01. REKAPITULASI JANUARI 2026.csv
4,02. REKAP FEBRUARI 2023.xlsx,02. REKAP FEBRUARI 2023,REKAPITULASI,OK,85,34,..\data\raw\rekapitulasi\02. REKAP FEBRUARI 2023.csv
5,02. REKAP FEBRUARI 2024.xlsx,02. REKAP FEBRUARI 2024,REKAPITULASI,OK,88,35,..\data\raw\rekapitulasi\02. REKAP FEBRUARI 2024.csv
6,02. REKAPITULASI FEBRUARI 2025.xlsx,02. REKAPITULASI FEBRUARI 2025,REKAPITULASI,OK,96,34,..\data\raw\rekapitulasi\02. REKAPITULASI FEBRUARI 2025.csv
7,02. REKAPITULASI FEBRUARI 2026.xlsx,02. REKAPITULASI FEBRUARI 2026,REKAPITULASI,OK,96,32,..\data\raw\rekapitulasi\02. REKAPITULASI FEBRUARI 2026.csv
8,03. REKAP MARET 2023.xlsx,03. REKAP MARET 2023,REKAPITULASI,OK,88,37,..\data\raw\rekapitulasi\03. REKAP MARET 2023.csv
9,03. REKAP MARET 2024.xlsx,03. REKAP MARET 2024,REKAPITULASI,OK,102,39,..\data\raw\rekapitulasi\03. REKAP MARET 2024.csv


In [22]:
# Pratinjau dan cek konsistensi layout
summary = []
rekap_data = {}   # local_name -> DataFrame mentah

for _, r in df_extract[df_extract["status"] == "OK"].iterrows():
    df = pd.read_csv(r["csv_path"], header=None, encoding="utf-8-sig", low_memory=False)
    rekap_data[r["local_name"]] = df

    # Buang baris dan kolom yang kosong semua untuk mengukur ukuran "isi" sebenarnya
    isi = df.dropna(how="all").dropna(axis=1, how="all")

    # Teks pertama yang muncul (biasanya judul laporan)
    first_cells = df.stack().astype(str).head(1)
    judul = first_cells.iloc[0][:60] if len(first_cells) else ""

    summary.append({
        "file": r["file"],
        "tab": r["tab"],
        "baris_mentah": df.shape[0],
        "kolom_mentah": df.shape[1],
        "baris_isi": isi.shape[0],
        "kolom_isi": isi.shape[1],
        "sel_terisi_%": round(df.notna().sum().sum() / df.size * 100, 1),
        "teks_pertama": judul,
    })

df_summary = pd.DataFrame(summary)
print(f"File terbaca: {len(df_summary)}")
print("\nSebaran jumlah kolom berisi:")
print(df_summary["kolom_isi"].value_counts().sort_index().to_string())
print("\nSebaran jumlah baris berisi:")
print(df_summary["baris_isi"].describe().round(1).to_string())

df_summary

File terbaca: 44

Sebaran jumlah kolom berisi:
kolom_isi
32     1
34     5
35     7
36     6
37    10
38    15

Sebaran jumlah baris berisi:
count    44.0
mean     93.4
std       4.3
min      80.0
25%      90.0
50%      95.0
75%      95.0
max      98.0


,file,tab,baris_mentah,kolom_mentah,baris_isi,kolom_isi,sel_terisi_%,teks_pertama
0,01. REKAP JANUARI 2023.xlsx,REKAPITULASI,82,37,80,37,66.5,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
1,01. REKAP JANUARI 2024.xlsx,REKAPITULASI,95,39,90,38,76.4,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
2,01. REKAPITULASI JANUARI 2025.xlsx,REKAPITULASI,96,37,95,37,68.1,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
3,01. REKAPITULASI JANUARI 2026.xlsx,REKAPITULASI,96,35,95,35,66.3,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
4,02. REKAP FEBRUARI 2023.xlsx,REKAPITULASI,85,34,83,34,83.4,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
5,02. REKAP FEBRUARI 2024.xlsx,REKAPITULASI,88,35,87,35,88.1,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
6,02. REKAPITULASI FEBRUARI 2025.xlsx,REKAPITULASI,96,34,95,34,67.9,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
7,02. REKAPITULASI FEBRUARI 2026.xlsx,REKAPITULASI,96,32,95,32,66.9,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
8,03. REKAP MARET 2023.xlsx,REKAPITULASI,88,37,87,37,81.4,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM
9,03. REKAP MARET 2024.xlsx,REKAPITULASI,102,39,97,38,56.4,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM


In [23]:
names = list(rekap_data.keys())
sample = [names[0], names[len(names) // 2], names[-1]]

for n in sample:
    print(f"\n{'=' * 70}\n{n} | shape: {rekap_data[n].shape}\n{'=' * 70}")
    display(rekap_data[n].head(15))


01. REKAP JANUARI 2023 | shape: (82, 37)


,0,1,2,3,4,5,6,7,8,9,...,27,28,29,30,31,32,33,34,35,36
0,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LAPORAN HARGA BAHAN POKOK DI KOTA SURABAYA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PERIODE JANUARI 2023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,,NAMA BAHAN POKOK DAN PENTING LAINNYA,SAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HET/HAPK (Harga Eceran Tertinggi),FLUKTUASI,NaN
4,NaN,NaN,NaN,1.00000,2.00000,3.00000,4.00000,5.00000,6.00000,7.00000,...,25.00000,26.00000,27.00000,28.00000,29.00000,30.00000,31.00000,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,BERAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,- Beras IR.64 Kwalitas I,Kg,12471.42857,12471.42857,12471.42857,12471.42857,12471.42857,12471.42857,12471.42857,...,12542.85714,12542.85714,12542.85714,12542.85714,12542.85714,12542.85714,12542.85714,12800,71.42857143,12499.07834
8,NaN,- Beras IR.64 Kwalitas II,Kg,11285.71429,11285.71429,11285.71429,11285.71429,11285.71429,11357.14286,11357.14286,...,11357.14286,11357.14286,11357.14286,11357.14286,11357.14286,11357.14286,11357.14286,NaN,71.42857143,11345.62212
9,NaN,- Beras IR.64 Kwalitas III,Kg,10325.00000,10325.00000,10325.00000,10325.00000,10325.00000,10491.66667,10491.66667,...,10491.66667,10491.66667,10491.66667,10491.66667,10491.66667,10491.66667,10491.66667,9450,166.6666667,10464.78495



06. REKAPITULASI JUNI 2025 | shape: (96, 36)


,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,BAGIAN PEREKONOMIAN DAN SUMBER DAYA ALAM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LAPORAN HARGA BAHAN POKOK DI KOTA SURABAYA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PERIODE JUNI 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,,NAMA BAHAN POKOK DAN PENTING LAINNYA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,HET/HAPK (Harga Eceran Tertinggi),FLUKTUASI,NaN,NaN
4,NaN,NaN,1,2,3,4,5,6,7,8,...,25,26,27,28,29,30,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,BERAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,- Beras Premium,15583.333333333334,15583.333333333334,15583.333333333334,15583.333333333334,15583.333333333334,15583.333333333334,15583.333333333334,15583.333333333334,...,15333.333333333334,15333.333333333334,15333.333333333334,15333.333333333334,15333.333333333334,15333.333333333334,14900,-250,15491.666667,NaN
8,NaN,- Beras IR.64 Medium Bulog,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,NaN,NaN,NaN,NaN
9,NaN,- Beras IR.64 Medium,13500,13500,13500,13500,13500,13500,13500,13500,...,13500,13500,13500,13500,13500,13500,12500,0,13500.000000,



12. REKAPITULASI DESEMBER 2025 | shape: (95, 36)


,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,LAPORAN HARGA BAHAN POKOK DI KOTA SURABAYA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,PERIODE DESEMBER 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,,NAMA BAHAN POKOK DAN PENTING LAINNYA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Harga Rata-Rata,FLUKTUASI,NaN
3,NaN,NaN,1.00000,2.00000,3.00000,4.00000,5.00000,6.00000,7.00000,8.00000,...,25.00000,26.00000,27.00000,28.00000,29.00000,30.00000,31.00000,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,BERAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,- Beras Premium,15714.28571,15714.28571,15714.28571,15714.28571,15714.28571,15642.85714,15642.85714,15571.42857,...,15642.85714,15642.85714,15642.85714,15642.85714,15814.28571,15814.28571,15814.28571,15664.0553,NaN,15664.0553
7,NaN,- Beras Medium Bulog,12250.00000,12250.00000,12250.00000,12250.00000,12250.00000,12250.00000,12250.00000,12250.00000,...,12250.00000,12250.00000,12250.00000,12250.00000,12250.00000,12250.00000,12300.00000,12251.6129,NaN,12251.6129
8,NaN,- Beras Medium,14166.66667,14166.66667,14166.66667,14166.66667,14166.66667,14166.66667,14166.66667,14166.66667,...,14166.66667,14166.66667,14166.66667,14166.66667,14500.00000,14500.00000,14500.00000,14198.92473,NaN,14198.92473
9,NaN,- Beras Jagung,11500.00000,11500.00000,11500.00000,11500.00000,11500.00000,11500.00000,11500.00000,11500.00000,...,11500.00000,11500.00000,11500.00000,11500.00000,11500.00000,11500.00000,11428.57143,11497.69585,NaN,11497.69585
